# Advantage Actor-Critic (A2C) Policy Gradient Agent
### LunarLander-v3 — Comparative RL Capstone

This notebook implements an **Advantage Actor-Critic (A2C)** agent as the natural evolution of the REINFORCE policy gradient method. Where REINFORCE suffers from high-variance Monte Carlo gradient estimates, A2C introduces a learned **critic** (value function) to compute per-step **advantage estimates**, dramatically reducing variance while maintaining the policy gradient framework.

**Key differences from REINFORCE:**
- Per-step bootstrapped updates (no need to wait for episode completion)
- Learned value baseline reduces gradient variance
- Advantage = TD error = `r + γV(s') - V(s)` replaces normalized returns
- Separate actor (policy) and critic (value) losses
- Entropy bonus preserved for exploration

## Import Libraries

In [ ]:
import gymnasium as gym
import numpy as np
import random
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
from collections import deque
import matplotlib.pyplot as plt
from IPython.display import clear_output

print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Initialize Environment

In [ ]:
env_name = "LunarLander-v3"
env = gym.make(env_name)

print("Environment:", env_name)
print("Observation Space:", env.observation_space)
print("  Shape:", env.observation_space.shape)
print("Action Space:", env.action_space)
print("  N:", env.action_space.n)

## Define Base Agent

In [ ]:
class Agent:
    """Base agent class — provides random action fallback and environment introspection."""
    def __init__(self, env):
        self.is_discrete = type(env.action_space) == gym.spaces.discrete.Discrete

        if self.is_discrete:
            self.action_size = env.action_space.n
        else:
            self.action_space_low = env.action_space.low
            self.action_space_high = env.action_space.high
            self.action_shape = env.action_space.shape

    def get_action(self, observation):
        """Random action (used for exploration fallback)."""
        if self.is_discrete:
            return random.choice(range(self.action_size))
        else:
            return np.random.uniform(
                self.action_space_low, self.action_space_high, self.action_shape
            )

## Actor-Critic Network

A **shared-trunk** architecture where the actor (policy head) and critic (value head) share early feature layers. This is more parameter-efficient and helps the critic's learned features inform the actor's policy.

In [ ]:
class ActorCriticNetwork(nn.Module):
    """
    Shared-trunk Actor-Critic network.
    
    Architecture:
        Input (state_size) → 128 → ReLU → 128 → ReLU
            ├─ Actor head  → action_size (log-softmax policy)
            └─ Critic head → 1 (state value V(s))
    """
    def __init__(self, state_size, action_size):
        super(ActorCriticNetwork, self).__init__()

        # Shared feature trunk
        self.shared = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )

        # Actor head: outputs action log-probabilities
        self.actor_head = nn.Linear(128, action_size)

        # Critic head: outputs scalar state value V(s)
        self.critic_head = nn.Linear(128, 1)

    def forward(self, x):
        features = self.shared(x)
        action_logits = self.actor_head(features)
        state_value = self.critic_head(features)
        return action_logits, state_value

    def get_action_and_value(self, state):
        """Returns action distribution, sampled action, log_prob, entropy, and V(s)."""
        action_logits, state_value = self.forward(state)
        dist = Categorical(logits=action_logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        return action.item(), log_prob, entropy, state_value.squeeze(-1)

## A2C Policy Gradient Agent

Core A2C update rule per step:

$$\text{Advantage} \; A(s,a) = r + \gamma V(s') - V(s)$$

$$\mathcal{L}_{\text{actor}} = -\log\pi(a|s) \cdot A(s,a)$$

$$\mathcal{L}_{\text{critic}} = \frac{1}{2}(r + \gamma V(s') - V(s))^2$$

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{actor}} + c_v \mathcal{L}_{\text{critic}} - c_e H(\pi)$$

In [ ]:
class PolicyGradientAgent(Agent):
    """
    Advantage Actor-Critic (A2C) Policy Gradient Agent.

    Upgrades over REINFORCE:
        - Per-step TD(0) advantage instead of Monte Carlo returns
        - Learned value baseline (critic) reduces gradient variance
        - Shared-trunk network for parameter efficiency
        - Entropy bonus for sustained exploration
        - Gradient clipping for training stability
    """

    def __init__(self, env, learning_rate=3e-4, discount_rate=0.99,
                 entropy_coeff=0.01, value_loss_coeff=0.5,
                 max_grad_norm=0.5):
        super().__init__(env)

        # Environment dimensions
        self.state_size = env.observation_space.shape[0]
        self.gamma = discount_rate

        # Loss weighting coefficients
        self.entropy_coeff = entropy_coeff       # c_e: encourages exploration
        self.value_loss_coeff = value_loss_coeff  # c_v: scales critic loss
        self.max_grad_norm = max_grad_norm        # gradient clipping threshold

        # Actor-Critic network (shared trunk)
        self.model = ActorCriticNetwork(self.state_size, self.action_size).to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)

        print("=== A2C PolicyGradientAgent Initialized ===")
        print(f"  State size:       {self.state_size}")
        print(f"  Action size:      {self.action_size}")
        print(f"  Learning rate:    {learning_rate}")
        print(f"  Discount (γ):     {self.gamma}")
        print(f"  Entropy coeff:    {self.entropy_coeff}")
        print(f"  Value loss coeff: {self.value_loss_coeff}")
        print(f"  Grad clip norm:   {self.max_grad_norm}")
        print(f"  Device:           {device}")
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"  Total parameters: {total_params:,}")
        print("===========================================")

    def get_action(self, state):
        """
        Sample action from the learned policy π(a|s).
        Returns: (action, log_prob, entropy, V(s))
        """
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        self.model.eval()
        with torch.no_grad():
            action_logits, state_value = self.model(state_tensor)
        self.model.train()

        dist = Categorical(logits=action_logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()

        return action.item(), log_prob, entropy, state_value.squeeze()

    def train_step(self, state, action, reward, next_state, done):
        """
        Single A2C update step using TD(0) advantage.

        Advantage A(s,a) = r + γ·V(s') - V(s)
        Actor loss  = -log π(a|s) · A(s,a)
        Critic loss = 0.5 · A(s,a)²
        Total loss  = actor_loss + c_v·critic_loss - c_e·entropy

        Returns dict with: actor_loss, critic_loss, entropy, advantage, total_loss
        """
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0).to(device)
        reward_tensor = torch.FloatTensor([reward]).to(device)

        # Forward pass: get policy distribution and value for current state
        action_logits, state_value = self.model(state_tensor)
        state_value = state_value.squeeze()

        # Critic bootstrap: V(s') — detached from actor graph
        with torch.no_grad():
            _, next_state_value = self.model(next_state_tensor)
            next_state_value = next_state_value.squeeze()
            # TD target: r + γ·V(s')  (V(s')=0 if terminal)
            td_target = reward_tensor + self.gamma * next_state_value * (1 - int(done))

        # Advantage (TD error): A(s,a) = td_target - V(s)
        advantage = (td_target - state_value).detach()

        # Actor loss: -log π(a|s) · A(s,a)
        dist = Categorical(logits=action_logits)
        log_prob = dist.log_prob(torch.tensor(action).to(device))
        entropy = dist.entropy()
        actor_loss = -(log_prob * advantage)

        # Critic loss: MSE between V(s) and TD target
        critic_loss = F.mse_loss(state_value, td_target.detach())

        # Combined loss
        total_loss = (
            actor_loss
            + self.value_loss_coeff * critic_loss
            - self.entropy_coeff * entropy
        )

        # Backprop + gradient clipping
        self.optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
        self.optimizer.step()

        return {
            "actor_loss": actor_loss.item(),
            "critic_loss": critic_loss.item(),
            "entropy": entropy.item(),
            "advantage": advantage.item(),
            "total_loss": total_loss.item()
        }

## Training Metrics Tracker

In [ ]:
class TrainingMetrics:
    """
    Tracks per-episode training metrics for the A2C agent.
    
    Recorded per episode:
        - Episode reward (sum of rewards)
        - Actor loss (mean over steps)
        - Critic loss (mean over steps)
        - Entropy (mean over steps)
        - Advantage (mean absolute over steps)
        - Episode length (number of steps)
    
    Provides rolling averages and comparison-ready plotting.
    """

    def __init__(self, rolling_window=100):
        self.rolling_window = rolling_window

        # Per-episode histories
        self.rewards = []
        self.actor_losses = []
        self.critic_losses = []
        self.entropies = []
        self.advantages = []
        self.episode_lengths = []

        # Accumulators for current episode
        self._ep_reward = 0
        self._ep_actor_losses = []
        self._ep_critic_losses = []
        self._ep_entropies = []
        self._ep_advantages = []
        self._ep_steps = 0

    def step(self, reward, train_info):
        """Record a single training step within an episode."""
        self._ep_reward += reward
        self._ep_steps += 1
        self._ep_actor_losses.append(train_info["actor_loss"])
        self._ep_critic_losses.append(train_info["critic_loss"])
        self._ep_entropies.append(train_info["entropy"])
        self._ep_advantages.append(abs(train_info["advantage"]))

    def end_episode(self):
        """Flush accumulators into episode-level histories."""
        self.rewards.append(self._ep_reward)
        self.actor_losses.append(np.mean(self._ep_actor_losses) if self._ep_actor_losses else 0)
        self.critic_losses.append(np.mean(self._ep_critic_losses) if self._ep_critic_losses else 0)
        self.entropies.append(np.mean(self._ep_entropies) if self._ep_entropies else 0)
        self.advantages.append(np.mean(self._ep_advantages) if self._ep_advantages else 0)
        self.episode_lengths.append(self._ep_steps)

        # Reset accumulators
        self._ep_reward = 0
        self._ep_actor_losses = []
        self._ep_critic_losses = []
        self._ep_entropies = []
        self._ep_advantages = []
        self._ep_steps = 0

    def rolling_avg(self, data):
        """Compute rolling average with the configured window."""
        if len(data) < self.rolling_window:
            return [np.mean(data[:i+1]) for i in range(len(data))]
        return [
            np.mean(data[max(0, i - self.rolling_window + 1):i + 1])
            for i in range(len(data))
        ]

    def print_status(self, episode):
        """Print compact status line for current episode."""
        r = self.rewards[-1]
        avg_r = np.mean(self.rewards[-self.rolling_window:])
        al = self.actor_losses[-1]
        cl = self.critic_losses[-1]
        ent = self.entropies[-1]
        adv = self.advantages[-1]
        steps = self.episode_lengths[-1]
        print(
            f"Ep {episode:4d} | "
            f"Reward: {r:7.1f} (avg {avg_r:7.1f}) | "
            f"ActorL: {al:7.4f} | CriticL: {cl:7.4f} | "
            f"Entropy: {ent:.3f} | |Adv|: {adv:.3f} | "
            f"Steps: {steps}"
        )

    def plot(self, title_prefix="A2C"):
        """Generate a 6-panel training summary plot."""
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"{title_prefix} — Training Summary", fontsize=16, fontweight='bold')
        episodes = range(1, len(self.rewards) + 1)

        # 1. Episode Reward
        ax = axes[0, 0]
        ax.plot(episodes, self.rewards, alpha=0.3, color='steelblue', label='Raw')
        ax.plot(episodes, self.rolling_avg(self.rewards), color='steelblue', linewidth=2, label=f'{self.rolling_window}-ep avg')
        ax.axhline(y=200, color='green', linestyle='--', alpha=0.5, label='Solved (200)')
        ax.set_title('Episode Reward')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Reward')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)

        # 2. Actor Loss
        ax = axes[0, 1]
        ax.plot(episodes, self.actor_losses, alpha=0.3, color='coral')
        ax.plot(episodes, self.rolling_avg(self.actor_losses), color='coral', linewidth=2)
        ax.set_title('Actor Loss (Policy Gradient)')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Loss')
        ax.grid(True, alpha=0.3)

        # 3. Critic Loss
        ax = axes[0, 2]
        ax.plot(episodes, self.critic_losses, alpha=0.3, color='mediumpurple')
        ax.plot(episodes, self.rolling_avg(self.critic_losses), color='mediumpurple', linewidth=2)
        ax.set_title('Critic Loss (Value Function)')
        ax.set_xlabel('Episode')
        ax.set_ylabel('MSE Loss')
        ax.grid(True, alpha=0.3)

        # 4. Entropy
        ax = axes[1, 0]
        ax.plot(episodes, self.entropies, alpha=0.3, color='seagreen')
        ax.plot(episodes, self.rolling_avg(self.entropies), color='seagreen', linewidth=2)
        ax.set_title('Policy Entropy')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Entropy (nats)')
        ax.grid(True, alpha=0.3)

        # 5. Mean |Advantage|
        ax = axes[1, 1]
        ax.plot(episodes, self.advantages, alpha=0.3, color='goldenrod')
        ax.plot(episodes, self.rolling_avg(self.advantages), color='goldenrod', linewidth=2)
        ax.set_title('Mean |Advantage| (TD Error Magnitude)')
        ax.set_xlabel('Episode')
        ax.set_ylabel('|A(s,a)|')
        ax.grid(True, alpha=0.3)

        # 6. Episode Length
        ax = axes[1, 2]
        ax.plot(episodes, self.episode_lengths, alpha=0.3, color='teal')
        ax.plot(episodes, self.rolling_avg(self.episode_lengths), color='teal', linewidth=2)
        ax.set_title('Episode Length')
        ax.set_xlabel('Episode')
        ax.set_ylabel('Steps')
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

## Instantiate Agent

In [ ]:
agent = PolicyGradientAgent(
    env,
    learning_rate=3e-4,
    discount_rate=0.99,
    entropy_coeff=0.01,
    value_loss_coeff=0.5,
    max_grad_norm=0.5
)

metrics = TrainingMetrics(rolling_window=100)

## Training Loop

A2C trains **per-step** (online), unlike REINFORCE which waits for the full episode. Each `(s, a, r, s', done)` transition produces an immediate gradient update through the TD advantage.

In [ ]:
NUM_EPISODES = 2000
PRINT_EVERY = 50
SOLVED_REWARD = 200  # LunarLander-v3 solved threshold

start_time = time.time()

for episode in range(1, NUM_EPISODES + 1):
    state, info = env.reset()
    done = False

    while not done:
        # Select action from policy
        action, _, _, _ = agent.get_action(state)

        # Step environment
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        # A2C per-step update
        train_info = agent.train_step(state, action, reward, next_state, done)

        # Record metrics
        metrics.step(reward, train_info)

        state = next_state

    # End-of-episode bookkeeping
    metrics.end_episode()

    if episode % PRINT_EVERY == 0:
        metrics.print_status(episode)

    # Early stopping: solved when 100-episode average >= 200
    if len(metrics.rewards) >= 100:
        recent_avg = np.mean(metrics.rewards[-100:])
        if recent_avg >= SOLVED_REWARD:
            elapsed = time.time() - start_time
            print(f"\n*** SOLVED at episode {episode}! ***")
            print(f"100-episode average: {recent_avg:.1f}")
            print(f"Training time: {elapsed:.1f}s")
            break

elapsed = time.time() - start_time
print(f"\nTraining complete. {len(metrics.rewards)} episodes in {elapsed:.1f}s")
print(f"Final 100-ep avg reward: {np.mean(metrics.rewards[-100:]):.1f}")

## Training Plots

In [ ]:
metrics.plot(title_prefix="A2C PolicyGradientAgent — LunarLander-v3")

## Evaluation — Watch the Trained Agent

In [ ]:
eval_env = gym.make(env_name)

NUM_EVAL_EPISODES = 20
eval_rewards = []

for ep in range(NUM_EVAL_EPISODES):
    state, _ = eval_env.reset()
    done = False
    total_reward = 0

    while not done:
        # Greedy action: use argmax of policy logits
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action_logits, _ = agent.model(state_tensor)
        action = torch.argmax(action_logits, dim=-1).item()

        state, reward, terminated, truncated, _ = eval_env.step(action)
        done = terminated or truncated
        total_reward += reward

    eval_rewards.append(total_reward)

eval_env.close()

print(f"Evaluation over {NUM_EVAL_EPISODES} episodes:")
print(f"  Mean reward:   {np.mean(eval_rewards):.1f}")
print(f"  Std reward:    {np.std(eval_rewards):.1f}")
print(f"  Min / Max:     {np.min(eval_rewards):.1f} / {np.max(eval_rewards):.1f}")

## Export Metrics for Cross-Method Comparison

This cell exports the A2C training curves so they can be loaded alongside DQN and REINFORCE results for the capstone comparison plots.

In [ ]:
a2c_results = {
    "method": "A2C",
    "rewards": metrics.rewards,
    "actor_losses": metrics.actor_losses,
    "critic_losses": metrics.critic_losses,
    "entropies": metrics.entropies,
    "advantages": metrics.advantages,
    "episode_lengths": metrics.episode_lengths,
    "eval_rewards": eval_rewards
}

# Optionally save to file
# import json
# with open("a2c_results.json", "w") as f:
#     json.dump({k: v if not isinstance(v, np.ndarray) else v.tolist() for k, v in a2c_results.items()}, f)

print("Results dict ready for comparison.")
print(f"  Episodes trained: {len(metrics.rewards)}")
print(f"  Final 100-ep avg: {np.mean(metrics.rewards[-100:]):.1f}")

## A2C vs REINFORCE vs DQN — Comparison Plot Scaffold

Paste in `dqn_results` and `reinforce_results` dicts from the integrated notebook to generate the full comparison.

In [ ]:
def plot_comparison(results_list, metric_key="rewards", ylabel="Reward",
                    title="Training Reward Comparison", rolling_window=100,
                    solved_line=200):
    """
    Overlay training curves from multiple methods.
    
    Args:
        results_list: list of dicts, each with 'method' and metric_key
        metric_key: which metric to plot
        ylabel: y-axis label
        title: plot title
    """
    colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']
    fig, ax = plt.subplots(figsize=(12, 6))

    for i, result in enumerate(results_list):
        data = result[metric_key]
        eps = range(1, len(data) + 1)
        color = colors[i % len(colors)]

        ax.plot(eps, data, alpha=0.15, color=color)

        # Rolling average
        rolling = []
        for j in range(len(data)):
            window = data[max(0, j - rolling_window + 1):j + 1]
            rolling.append(np.mean(window))
        ax.plot(eps, rolling, color=color, linewidth=2,
                label=f"{result['method']} ({rolling_window}-ep avg)")

    if solved_line is not None:
        ax.axhline(y=solved_line, color='green', linestyle='--', alpha=0.5, label=f'Solved ({solved_line})')

    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Episode')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Example usage (uncomment when other results are available):
# plot_comparison([a2c_results, dqn_results, reinforce_results])

# For now, plot A2C alone:
plot_comparison([a2c_results], title="A2C Training Reward — LunarLander-v3")

## Cleanup

In [ ]:
env.close()
print("Environment closed.")